# 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from google.colab import files
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim
import copy

# 2. Load Stock Data

In [5]:
files = files.upload()
Stock_Data = pd.read_csv("nvda_us_d (5).csv")

Saving nvda_us_d (5).csv to nvda_us_d (5) (1).csv


# 3. Feature Engineering


In [6]:
Stock_Data['MA_5'] = Stock_Data['Close'].rolling(5).mean()
Stock_Data['MA_10'] = Stock_Data['Close'].rolling(10).mean()
Stock_Data['MA_20'] = Stock_Data['Close'].rolling(20).mean()
Stock_Data['MA_20_ratio'] = (Stock_Data['Close'] / Stock_Data['MA_20']) - 1
Stock_Data['Close_t-1'] = Stock_Data['Close'].shift(1)
Stock_Data['Close_t-2'] = Stock_Data['Close'].shift(2)
Stock_Data['Close_t-3'] = Stock_Data['Close'].shift(3)
Stock_Data['Close_t-4'] = Stock_Data['Close'].shift(4)
Stock_Data['Close_t-5'] = Stock_Data['Close'].shift(5)

In [7]:
delta = Stock_Data['Close'].diff()

gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)

avg_gains = gains.rolling(14).mean()
avg_losses = losses.rolling(14).mean()

RS = (avg_gains / avg_losses)
Stock_Data["RSI_14"] = 100 - (100 / (1 + RS))

In [8]:
EMA_12 = Stock_Data['Close'].ewm(span=12,adjust=False).mean()
EMA_26 = Stock_Data['Close'].ewm(span=26,adjust=False).mean()
Stock_Data['MACD'] = EMA_12-EMA_26
Stock_Data['MACD_Signal'] = Stock_Data['MACD'].ewm(span=9,adjust=False).mean()
Stock_Data['MACD_Histogram'] = Stock_Data['MACD_Signal'] - Stock_Data['MACD']
Stock_Data.reset_index(drop=True, inplace=True)
Stock_Data.ffill(inplace=True)
Stock_Data['Return'] = Stock_Data['Close'].pct_change()*100
Stock_Data['Return_t-1'] = Stock_Data['Return'].shift(1)
Stock_Data['Volume_Ratio'] = Stock_Data['Volume'] / Stock_Data['Volume'].rolling(20).mean()
Stock_Data['Volatility_20'] = Stock_Data['Return'].rolling(20).std()

# 4. Define the Prediction Target

In [9]:
Stock_Data['Next_5_day_cumulative_return'] = ((Stock_Data['Close'].shift(-5)/Stock_Data['Close']) - 1) * 100
Stock_Data = Stock_Data.dropna()

# 5. Create LSTM Sequences

In [10]:
window_size = 60
X = []
y = []
features = ['MA_20_ratio','RSI_14','Return','MACD_Histogram','Return_t-1','Volume_Ratio','Volatility_20']
for i in range(len(Stock_Data)-window_size):
  X_window = Stock_Data[features].iloc[i:i+window_size]
  y_target = Stock_Data['Next_5_day_cumulative_return'].iloc[i+window_size]

  X.append(X_window)
  y.append(y_target)

X = np.array(X)
y = np.array(y).reshape(-1,1)

# 6. Split the Data Chronologically

In [11]:
split = int(len(X) * 0.6)
split_2 = int(len(X)*0.8)
gap = 5

X_train = X[:split-gap]
y_train = y[:split-gap]

X_val = X[split:split_2-gap]
y_val = y[split:split_2-gap]

X_test = X[split_2:]
y_test = y[split_2:]

In [12]:
original_train_shape = X_train.shape
X_train = X_train.reshape(-1,X_train.shape[2])

original_test_shape = X_test.shape
X_test = X_test.reshape(-1,X_test.shape[2])

original_val_shape = X_val.shape
X_val = X_val.reshape(-1,X_val.shape[2])

# 7. Scale the Features

In [13]:
Standard_Scaler = StandardScaler()
y_Scaler = StandardScaler()

X_train = Standard_Scaler.fit_transform(X_train)
X_test = Standard_Scaler.transform(X_test)
X_val = Standard_Scaler.transform(X_val)

y_train_scaled = y_Scaler.fit_transform(y_train)
y_val_scaled = y_Scaler.transform(y_val)
y_test_scaled = y_Scaler.transform(y_test)

In [14]:
X_train = X_train.reshape(original_train_shape)
X_test = X_test.reshape(original_test_shape)
X_val = X_val.reshape(original_val_shape)

# 8. Convert Data to PyTorch Tensors

In [15]:
X_train = torch.from_numpy(X_train).float()
X_test = torch.from_numpy(X_test).float()
X_val = torch.from_numpy(X_val).float()

y_train = torch.from_numpy(y_train_scaled).float()
y_val = torch.from_numpy(y_val_scaled).float()
y_test = torch.from_numpy(y_test_scaled).float()


# 9. Create the PyTorch Dataset and DataLoader

In [16]:
class StockDataSet(Dataset):

  def __init__(self,X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self,index):
    return self.X[index], self.y[index]

In [17]:
train_dataset = StockDataSet(X_train,y_train)
test_dataset = StockDataSet(X_test,y_test)

In [18]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_dataset,batch_size=20,shuffle=False)

# 10. Build the LSTM Model

In [19]:
class LSTM_Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.lstm = nn.LSTM(
    input_size = 7,
    hidden_size = 8,
    num_layers = 2,
    dropout=0.5,
    batch_first = True)
    self.fc = nn.Linear(8,1)

  def forward(self,x):
    output,_ = self.lstm(x)
    output = output[:,-1,:]
    output = self.fc(output)
    return output

In [20]:
torch.manual_seed(42)
LSTM = LSTM_Model()
loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(params=LSTM.parameters(),lr=0.0005,weight_decay=0.01)

# 11. Train the LSTM

In [21]:
epochs = 50
best_val_loss = float("inf")


for epoch in range(epochs):

    total_train_loss = 0

    LSTM.train()

    for X_batch, y_batch in train_dataloader:

        y_pred = LSTM(X_batch)

        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_dataloader)

    LSTM.eval()

    with torch.inference_mode():
        val_preds = LSTM(X_val)
        val_loss = loss_fn(val_preds, y_val)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_model = copy.deepcopy(LSTM.state_dict())

    if epoch % 10 == 0:
        print("Epoch:", epoch)
        print("Training loss:", average_train_loss)
        print("Validation loss:", val_loss.item())

print("Epoch: ",epoch)
print("Training loss:", average_train_loss)
print("Validation loss:", val_loss.item())

Epoch: 0
Training loss: 0.7981337274496372
Validation loss: 0.4295482337474823
Epoch: 10
Training loss: 0.7885540448702298
Validation loss: 0.4378203749656677
Epoch: 20
Training loss: 0.7879094183444977
Validation loss: 0.4396723210811615
Epoch: 30
Training loss: 0.7800414367364004
Validation loss: 0.44051164388656616
Epoch: 40
Training loss: 0.779795854137494
Validation loss: 0.44086530804634094
Epoch:  49
Training loss: 0.765332852418606
Validation loss: 0.4427002966403961


# 12. Evaluate the Best Model on the Test Set

In [22]:
LSTM.load_state_dict(best_state_model)
LSTM.eval()
with torch.inference_mode():
    test_preds = LSTM(X_test)
    test_loss = loss_fn(test_preds, y_test)

print("LSTM Test Loss:", test_loss.item())

LSTM Test Loss: 0.5626528263092041


# 13. Compare the LSTM Against a Baseline

In [23]:
baseline_preds = torch.zeros_like(y_test)
baseline_loss = loss_fn(baseline_preds, y_test)

print("BaseLine Loss: ",baseline_loss.item())
print("LSTM Test loss: ",test_loss.item())

BaseLine Loss:  0.5682141780853271
LSTM Test loss:  0.5626528263092041


In [24]:
predictions_actual_pct = y_Scaler.inverse_transform(test_preds.cpu().numpy())
y_test_actual_pct = y_Scaler.inverse_transform(y_test.cpu().numpy())

# 14. Make Predictions

In [25]:
latest_data = Stock_Data[features].tail(60).values


latest_scaled = Standard_Scaler.transform(latest_data)

live_tensor = torch.tensor(latest_scaled).float().unsqueeze(0)

LSTM.eval()
with torch.inference_mode():
    raw_pred = LSTM(live_tensor).numpy()

final_prediction = y_Scaler.inverse_transform(raw_pred)[0][0]

print("Predicted Return for the next 5 days:",final_prediction,"%")

Predicted Return for the next 5 days: 1.31476 %
